In [ ]:
import pyEDM
import pandas as pd
import numpy as np
import random

p22 = pd.read_csv('../data/processed_22.csv')
p23 = pd.read_csv('../data/processed_23.csv')
p24 = pd.read_csv('../data/processed_24.csv')
p25 = pd.read_csv('../data/processed_25.csv')
combined = pd.concat([p22, p23, p24, p25], ignore_index=True)

In [ ]:
env = pd.read_csv('../data/environment_all.csv')
env = env.drop(columns=['fluorescent_dissolved_organic_matter_eco', 'sea_water_turbidity_eco', 'waterlevel_predicted_m'])

# ones of contention (due to too many missing values):
    # mass_concentration_of_oxygen_in_sea_water_seaphox
    # mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox
    # fractional_saturation_of_oxygen_in_sea_water_seaphox
    # sea_water_ph_reported_on_total_scale_seaphox_external

env = env.drop(columns=['mass_concentration_of_oxygen_in_sea_water_seaphox', 'mole_concentration_of_dissolved_molecular_oxygen_in_sea_water_seaphox', 'fractional_saturation_of_oxygen_in_sea_water_seaphox', 'sea_water_ph_reported_on_total_scale_seaphox_external'])
env.head()

In [ ]:
features = [
    "Lpoly_expected_ml", "Area", "Biovolume", "MajorAxisLength",
    # "MinorAxisLength", "Perimeter", "Orientation", "Eccentricity",
    # "Solidity", "texture_uniformity", "texture_smoothness",
    # "texture_average_gray_level", "texture_entropy",
    # "texture_average_contrast", "H90", "H180", "Hflip",
    # "Extent", "EquivDiameter", "ConvexArea", "ConvexPerimeter"
]

df = combined[["date"] + features].copy()
df["date"] = pd.to_datetime(df["date"].astype(str), format="%Y%m%d")
df = df.sort_values("date").set_index("date")

# Force shared daily index
df = df.asfreq("D")

df_filled = df.copy()
was_imputed = pd.DataFrame(index=df.index)

for col in features:
    ema = df[col].ewm(span=30, adjust=False).mean()
    df_filled[col] = df[col].fillna(ema)
    was_imputed[col] = df[col].isna()

# normalize features
df_norm = df_filled.copy()
for col in features:
    mu = df_filled[col].mean()
    sigma = df_filled[col].std()
    
    df_norm[col] = (df_filled[col] - mu) / sigma

predictors = [col for col in features if col != "Lpoly_expected_ml"]

df_mv = df_norm.copy()
df_mv = df_mv.reset_index()
df_mv["t"] = np.arange(1, len(df_mv) + 1)

df_mv = df_mv[["t", "Lpoly_expected_ml"] + predictors]

In [ ]:
res_mv = pyEDM.Multiview(
    dataFrame=df_mv,
    columns=predictors,
    target="Lpoly_expected_ml",
    E=4,
    tau=1,
    Tp=1,
    lib=f"1 {len(df_mv)}",
    pred=f"1 {len(df_mv)}",
    D=3 # dimension 
)

obs = res_mv["Observations"].to_numpy()
pred = res_mv["Predictions"].to_numpy()

mask = np.isfinite(obs) & np.isfinite(pred)

rho_mv = np.corrcoef(obs[mask], pred[mask])[0, 1]
print("Multiview rho:", rho_mv)

In [ ]:
res_mv.head()